# Mapping espacial unidades historicas

Código encargado de construir una tabla de correspondencia (mapping) entre las comunas modernas (2024) y las comunas históricas de 1970, basándose en ubicación geográfica de los LPA de 1970 y 2024.

In [2]:
import geopandas as gpd
import pandas as pd
import unicodedata
import re
import warnings

warnings.filterwarnings('ignore')

In [3]:
# RUTAS SHAPEFILES

# Shapefile moderno (2024 / DPA 2023)
PATH_SHP_MODERNO = '/Users/Angelo/Dropbox/Tesis 2026 ME/referencias historicas carreteras/Qgis proyects/DPA_2023/COMUNAS/COMUNAS_v1.shp'

# Shapefile histórico (1970)
PATH_SHP_1970 = '/Users/Angelo/Dropbox/DPA Historica/Comunas_1970.shp'

# Output
PATH_OUT = '/Users/Angelo/Dropbox/Tesis 2026 ME/Codes/mapping_1970_2024_v2.csv'

In [6]:

def normalizar(txt):
    if pd.isna(txt): 
        return ''
    txt = str(txt).strip().upper()
    txt = unicodedata.normalize('NFD', txt)
    txt = ''.join(c for c in txt if unicodedata.category(c) != 'Mn')
    txt = re.sub(r"[^A-Z0-9\' ]", '', txt).strip()
    return txt

# CARGA Y PREPARACIÓN GEOGRÁFICA
shp_moderno = gpd.read_file(PATH_SHP_MODERNO)
shp_1970 = gpd.read_file(PATH_SHP_1970)

# Asegurar que ambos shapefiles estén en el mismo sistema de coordenadas
shp_moderno = shp_moderno.to_crs(shp_1970.crs)

# Arreglar problema de codificación (encoding) en el shapefile de 1970 (latin-1 a utf-8)
shp_1970['COMUNA'] = shp_1970['COMUNA'].apply(
    lambda x: str(x).encode('latin-1').decode('utf-8') if isinstance(x, str) else x
)

# Extraer los centroides de las comunas modernas
centroids_modernos = shp_moderno.copy()
centroids_modernos['geometry'] = centroids_modernos.geometry.centroid


In [7]:

#  SPATIAL JOIN
# Busca dentro de qué polígono histórico (1970) cae el centroide moderno (2024)
sjoin = gpd.sjoin(centroids_modernos, shp_1970, how='left', predicate='within')


# Mantener exactamente las mismas columnas que el archivo original
mapping_df = pd.DataFrame({
    'daughter_2024': sjoin['COMUNA_left'].values,
    'index_right': sjoin['index_right'].values if 'index_right' in sjoin.columns else None,
    'mother_1970': sjoin['COMUNA_right'].values,
})

# Generar las columnas normalizadas
mapping_df['mother_1970_norm'] = mapping_df['mother_1970'].apply(normalizar)
mapping_df['daughter_2024_norm'] = mapping_df['daughter_2024'].apply(normalizar)

In [ ]:
#  EXPORTAR
mapping_df.to_csv(PATH_OUT, index=False)
huerfanas = mapping_df[mapping_df['mother_1970'].isna()]
()